<a href="https://colab.research.google.com/github/chipojaya1/ISS-Economic-Distance/blob/main/Economic_distance_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###1 Prepare environment and load the data

In [257]:
# Load libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
import plotly.express as px
import plotly.graph_objects as go

from scipy.spatial import distance
sns.set_theme(style="whitegrid", context="notebook")

In [258]:
# Import wdi data as wdi from my github respository
df = pd.read_excel('https://github.com/chipojaya1/ISS-Economic-Distance/blob/main/data/WDI_Raw_2019.xlsx?raw=true')
print("df type:", type(df))
print("df shape:", df.shape)
df.info()
df.head()

df type: <class 'pandas.core.frame.DataFrame'>
df shape: (873, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 873 entries, 0 to 872
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Country Name   870 non-null    object
 1   Country Code   868 non-null    object
 2   Series Name    868 non-null    object
 3   Series Code    868 non-null    object
 4   2019 [YR2019]  868 non-null    object
dtypes: object(5)
memory usage: 34.2+ KB


,Country Name,Country Code,Series Name,Series Code,2019 [YR2019]
0,Afghanistan,AFG,GDP per capita (constant 2015 US$),NY.GDP.PCAP.KD,557.861533
1,Afghanistan,AFG,"Inflation, GDP deflator (annual %)",NY.GDP.DEFL.KD.ZG,6.52148
2,Afghanistan,AFG,Imports of goods and services (% of GDP),NE.IMP.GNFS.ZS,..
3,Afghanistan,AFG,Exports of goods and services (% of GDP),NE.EXP.GNFS.ZS,..
4,Albania,ALB,GDP per capita (constant 2015 US$),NY.GDP.PCAP.KD,5072.436484


In [259]:
# 1. Clean up potential leading/trailing whitespaces in string columns
df['Series Name'] = df['Series Name'].str.strip()
display(df.head())

,Country Name,Country Code,Series Name,Series Code,2019 [YR2019]
0,Afghanistan,AFG,GDP per capita (constant 2015 US$),NY.GDP.PCAP.KD,557.861533
1,Afghanistan,AFG,"Inflation, GDP deflator (annual %)",NY.GDP.DEFL.KD.ZG,6.52148
2,Afghanistan,AFG,Imports of goods and services (% of GDP),NE.IMP.GNFS.ZS,..
3,Afghanistan,AFG,Exports of goods and services (% of GDP),NE.EXP.GNFS.ZS,..
4,Albania,ALB,GDP per capita (constant 2015 US$),NY.GDP.PCAP.KD,5072.436484


In [260]:
# 2. Define the mapping for Series Name renaming
name_mapping = {
    'GDP per capita (constant 2015 US$)': 'gdp_per_capita',
    'Inflation, GDP deflator (annual %)': 'inflation',
    'Imports of goods and services (% of GDP)': 'imports',
    'Exports of goods and services (% of GDP)': 'exports'
}

# Filter to keep only the series we are interested in and map their names
df_filtered = df[df['Series Name'].isin(name_mapping.keys())].copy()
df_filtered['Series Name'] = df_filtered['Series Name'].map(name_mapping)
display(df_filtered.head())

,Country Name,Country Code,Series Name,Series Code,2019 [YR2019]
0,Afghanistan,AFG,gdp_per_capita,NY.GDP.PCAP.KD,557.861533
1,Afghanistan,AFG,inflation,NY.GDP.DEFL.KD.ZG,6.52148
2,Afghanistan,AFG,imports,NE.IMP.GNFS.ZS,..
3,Afghanistan,AFG,exports,NE.EXP.GNFS.ZS,..
4,Albania,ALB,gdp_per_capita,NY.GDP.PCAP.KD,5072.436484


In [261]:
# 3. Clean the value column: replace '..' with NaN and convert to float
val_col = '2019 [YR2019]'
df_filtered[val_col] = df_filtered[val_col].replace('..', np.nan)
df_filtered[val_col] = pd.to_numeric(df_filtered[val_col], errors='coerce')
display(df_filtered.head())

/tmp/ipykernel_601/3241800311.py:3: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



,Country Name,Country Code,Series Name,Series Code,2019 [YR2019]
0,Afghanistan,AFG,gdp_per_capita,NY.GDP.PCAP.KD,557.861533
1,Afghanistan,AFG,inflation,NY.GDP.DEFL.KD.ZG,6.521480
2,Afghanistan,AFG,imports,NE.IMP.GNFS.ZS,NaN
3,Afghanistan,AFG,exports,NE.EXP.GNFS.ZS,NaN
4,Albania,ALB,gdp_per_capita,NY.GDP.PCAP.KD,5072.436484


In [262]:
# 4. Pivot from long to wide (short) format
df_wide = df_filtered.pivot(index=['Country Name', 'Country Code'], columns='Series Name', values=val_col).reset_index()

# Reset column index name
df_wide.columns.name = None

# Display the cleaned and reshaped DataFrame
print("df_wide shape:", df_wide.shape)
display(df_wide.head())

df_wide shape: (217, 6)


,Country Name,Country Code,exports,gdp_per_capita,imports,inflation
0,Afghanistan,AFG,NaN,557.861533,NaN,6.521480
1,Albania,ALB,30.936122,5072.436484,44.446006,1.000633
2,Algeria,DZA,20.407942,4672.664087,26.098901,0.882993
3,American Samoa,ASM,61.669243,12524.016025,94.899536,1.748289
4,Andorra,AND,NaN,39346.275036,NaN,1.374754


In [263]:
# rename column Country Name to country, drop country code and mask df as wdi_df
wdi_df = df_wide.rename(columns={'Country Name': 'country'}).drop(columns=['Country Code'])
print("wdi_df shape:", wdi_df.shape)
wdi_df.head()

wdi_df shape: (217, 5)


,country,exports,gdp_per_capita,imports,inflation
0,Afghanistan,NaN,557.861533,NaN,6.521480
1,Albania,30.936122,5072.436484,44.446006,1.000633
2,Algeria,20.407942,4672.664087,26.098901,0.882993
3,American Samoa,61.669243,12524.016025,94.899536,1.748289
4,Andorra,NaN,39346.275036,NaN,1.374754


###2. Summarize the individual sub-components of the distance measure. Report and interpret the minimum, maximum, mean, median, standard deviation for each of the sub-components and the correlation between each of these sub-components

Explore summary statistics  and correlation coefficients



In [264]:
wdi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 217 entries, 0 to 216
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country         217 non-null    object 
 1   exports         183 non-null    float64
 2   gdp_per_capita  208 non-null    float64
 3   imports         183 non-null    float64
 4   inflation       209 non-null    float64
dtypes: float64(4), object(1)
memory usage: 8.6+ KB


In [265]:
# Summary statistics
summary_stats = wdi_df.describe()
print("Summary Statistics:")
display(summary_stats)

Summary Statistics:


,exports,gdp_per_capita,imports,inflation
count,183.000000,208.000000,183.000000,209.000000
mean,43.059452,17544.183111,48.085723,110.387032
std,31.529306,24664.928816,28.529602,1515.376731
min,7.779936,270.411006,10.607517,-4.603271
25%,22.282452,2423.171056,28.704516,0.848015
50%,36.400882,6533.112217,41.469590,2.375860
75%,52.400877,21751.331731,58.928664,4.175673
max,206.580146,185582.668516,176.092206,21909.342668


In [266]:
# find the number of missing values in each column
missing_values = wdi_df.isnull().sum()

# turn the missing value counts into percentages
missing_values_percentage = (missing_values / len(wdi_df)) * 100

# return counts and percentages of missing values in each column to 1 decimal place
missing_data = pd.DataFrame({'Missing Values': missing_values, 'Percentage(%)': missing_values_percentage.round(1)})
print("Missing Data:")
display(missing_data)

Missing Data:


,Missing Values,Percentage(%)
country,0,0.0
exports,34,15.7
gdp_per_capita,9,4.1
imports,34,15.7
inflation,8,3.7


Keeping rows with missing data will not be useful since we need to compare one data point to another.

In [267]:
# Drop rows with any missing values in the 3 indicators
cols = ['gdp_per_capita', 'inflation', 'imports', 'exports']
wdi_df = wdi_df.dropna(subset=cols)
print("wdi_df shape:", wdi_df.shape)
wdi_df.head()

wdi_df shape: (182, 5)


,country,exports,gdp_per_capita,imports,inflation
1,Albania,30.936122,5072.436484,44.446006,1.000633
2,Algeria,20.407942,4672.664087,26.098901,0.882993
3,American Samoa,61.669243,12524.016025,94.899536,1.748289
5,Angola,36.950082,3105.599712,23.421225,20.249856
6,Antigua and Barbuda,69.164695,18525.258891,68.373089,0.712335


In [268]:
wdi_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 182 entries, 1 to 216
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country         182 non-null    object 
 1   exports         182 non-null    float64
 2   gdp_per_capita  182 non-null    float64
 3   imports         182 non-null    float64
 4   inflation       182 non-null    float64
dtypes: float64(4), object(1)
memory usage: 8.5+ KB


In [269]:
# Keep a consistent style across all figures
cols = ["gdp_per_capita", "inflation", "imports", "exports"]

# Keep complete observations across all four indicators
plot_df = wdi_df.dropna(subset=cols).copy()

print(f"Economies included in analysis: {len(plot_df)}")

Economies included in analysis: 182


#### FIGURE 1: GDP PER CAPITA DISTRIBUTION

In [270]:
# Calculate summary metrics
mean_val = plot_df["gdp_per_capita"].mean()
median_val = plot_df["gdp_per_capita"].median()

# Create interactive Plotly Histogram
fig = px.histogram(
    plot_df,
    x="gdp_per_capita",
    nbins=28,
    color_discrete_sequence=["lightblue"],
    title="<b>GDP per capita varied widely in 2019</b><br><sup>2019 | 182 economies with complete data | constant 2015 US$ per person</sup>"
)

# Customize layout and add vertical lines for mean/median
fig.update_layout(
    xaxis_title="GDP per capita (constant 2015 US$)",
    yaxis_title="Number of economies",
    template="plotly_white",
    width=850,
    height=450,
    showlegend=False
)

# Add baseline axis lines
fig.update_xaxes(showline=True, linewidth=0.1, linecolor='lightgray', mirror=True)
fig.update_yaxes(showline=True, linewidth=0.1, linecolor='lightgray', mirror=True)

# Add Mean Line
fig.add_vline(
    x=mean_val,
    line_dash="dash",
    line_color="#C65D43",
    annotation_text=f"Mean: ${mean_val:,.0f}",
    annotation_position="top right"
)

# Add Median Line
fig.add_vline(
    x=median_val,
    line_dash="dashdot",
    line_color="#34495E",
    annotation_text=f"Median: ${median_val:,.0f}",
    annotation_position="top left"
)

fig.show()

GDP per capita varies greatly across economies. Most have relatively low economic output per person, while a much smaller group has considerably higher values. This difference highlights the unequal distribution of economic resources and productive capacity across the world.

#### FIGURE 2: Distribution of Inflation Across Economies (2019)

In [271]:
# Filter data for display purposes (retaining all observations in dataset)
inflation_display_min = -10
inflation_display_max = 30

inflation_visible = plot_df[plot_df["inflation"].between(inflation_display_min, inflation_display_max)].copy()
outside_count = len(plot_df) - len(inflation_visible)
overall_median = plot_df["inflation"].median()

# Create interactive Plotly Histogram
fig = px.histogram(
    inflation_visible,
    x="inflation",
    nbins=25,
    color_discrete_sequence=["lightblue"],
    title=f"<b>How Inflation Rates Differed Across Economies</b><br><sup>2019 | {len(inflation_visible)} of {len(plot_df)} economies shown ({inflation_display_min}% to {inflation_display_max}%)</sup>"
)

# Add vertical line for overall median
fig.add_vline(
    x=overall_median,
    line_dash="dash",
    line_color="#34495E",
    annotation_text=f"Overall median: {overall_median:.1f}%",
    annotation_position="top right"
)

# Customize layout
fig.update_layout(
    xaxis_title="GDP-deflator inflation (annual %)",
    yaxis_title="Number of economies",
    template="plotly_white",
    width=850,
    height=450,
    xaxis_range=[inflation_display_min, inflation_display_max]
)

# Add baseline axis lines
fig.update_xaxes(showline=True, linewidth=0.1, linecolor='lightgray', mirror=True)
fig.update_yaxes(showline=True, linewidth=0.1, linecolor='lightgray', mirror=True)

fig.show()
print(f"Note: {outside_count} of {len(plot_df)} economies fall outside the displayed range (-10% to 30%). All observations remain in the analysis.")

Note: 5 of 182 economies fall outside the displayed range (-10% to 30%). All observations remain in the analysis.


Most economies experienced relatively modest inflation in 2019. However, some experienced exceptionally high inflation rates. These extreme cases show that economic conditions differed substantially across economies, particularly in terms of price stability.

#### FIGURE 3: IMPORTS DISTRIBUTION

In [272]:
imports_mean = plot_df["imports"].mean()
imports_median = plot_df["imports"].median()

# Create interactive Plotly Histogram
fig = px.histogram(
    plot_df,
    x="imports",
    nbins=24,
    color_discrete_sequence=["lightblue"],
    title="<b>How Much Economies Import Relative to Their Size</b><br><sup>2019 | Imports of goods and services as a percentage of GDP</sup>"
)

# Add lines for mean and median
fig.add_vline(
    x=imports_mean,
    line_dash="dash",
    line_color="#C65D43",
    annotation_text=f"Mean: {imports_mean:.1f}%",
    annotation_position="top right"
)

fig.add_vline(
    x=imports_median,
    line_dash="dashdot",
    line_color="#34495E",
    annotation_text=f"Median: {imports_median:.1f}%",
    annotation_position="top left"
)

fig.update_layout(
    xaxis_title="Imports of goods and services (% of GDP)",
    yaxis_title="Number of economies",
    template="plotly_white",
    width=850,
    height=450,
    xaxis_range=[0, 220]
)

# Add baseline axis lines
fig.update_xaxes(showline=True, linewidth=0.1, linecolor='lightgray', mirror=True)
fig.update_yaxes(showline=True, linewidth=0.1, linecolor='lightgray', mirror=True)

fig.show()

Economies differ in how much they purchase from the rest of the world relative to the size of their economies. Some rely more heavily on international goods and services, while others import a smaller share of their GDP.

#### FIGURE 4: EXPORTS DISTRIBUTION

In [273]:
exports_mean = plot_df["exports"].mean()
exports_median = plot_df["exports"].median()

# Create interactive Plotly Histogram
fig = px.histogram(
    plot_df,
    x="exports",
    nbins=24,
    color_discrete_sequence=["lightblue"],
    title="<b>How Much Economies Export Relative to Their Size</b><br><sup>2019 | Exports of goods and services as a percentage of GDP</sup>"
)

# Add lines for mean and median
fig.add_vline(
    x=exports_mean,
    line_dash="dash",
    line_color="#C65D43",
    annotation_text=f"Mean: {exports_mean:.1f}%",
    annotation_position="top right"
)

fig.add_vline(
    x=exports_median,
    line_dash="dashdot",
    line_color="#34495E",
    annotation_text=f"Median: {exports_median:.1f}%",
    annotation_position="top left"
)

fig.update_layout(
    xaxis_title="Exports of goods and services (% of GDP)",
    yaxis_title="Number of economies",
    template="plotly_white",
    width=850,
    height=450,
    xaxis_range=[0, 220]
)

# Add baseline axis lines
fig.update_xaxes(showline=True, linewidth=0.1, linecolor='lightgray', mirror=True)
fig.update_yaxes(showline=True, linewidth=0.1, linecolor='lightgray', mirror=True)

fig.show()

Exports also vary considerably. Some economies sell a much larger share of their goods and services internationally than others. This illustrates differences in how strongly economies participate in international trade.

#### FIGURE 5: CORRELATION HEATMAP

In [274]:
import plotly.express as px

# Calculate Pearson correlation matrix
correlations = plot_df[cols].corr(method="pearson")

# Define friendly display labels
labels = ["GDP per capita", "Inflation", "Imports", "Exports"]

# Create an interactive Plotly Heatmap
fig = px.imshow(
    correlations,
    x=labels,
    y=labels,
    color_continuous_scale="BrBG",
    zmin=-1,
    zmax=1,
    text_auto=".2f",
    title="<b>How the Four Economic Indicators Relate</b>"
)

# Update layout with formatting matching the style requirements
fig.update_layout(
    title_x=0,
    width=600,
    height=550,
    coloraxis_colorbar=dict(title="Correlation"),
    margin=dict(l=40, r=40, t=80, b=40)
)

fig.show()

The heatmap shows how the four economic indicators relate to one another across economies in 2019.

The strongest relationship is between imports and exports (0.85). This suggests that economies that import more goods and services relative to their GDP also tend to export more. In other words, economies that are more engaged in international trade often have higher levels of both imports and exports.

GDP per capita has a moderate positive relationship with exports (0.48) and a weaker positive relationship with imports (0.27). This suggests that economies with higher economic output per person tend to have somewhat greater participation in international trade, particularly through exports.

Inflation has very weak relationships with the other three indicators. This suggests that differences in inflation do not closely follow differences in GDP per capita or trade activity across the economies studied.

A correlation of 0.85 does not necessarily mean trade causes economic growth or that importing causes exporting. It simply means those two indicators tend to move together across the economies in your dataset.

**Overall, the heatmap shows that the four indicators capture different aspects of economic conditions, although imports and exports are closely related.** This is important for the next stage of the analysis, where Mahalanobis distance is used to measure how economically different each economy is from the United States while accounting for relationships between the indicators.

**Do the actual values move together in a roughly straight-line pattern?** Pearson uses the numerical values directly and can be strongly influenced by exceptional observations, such as Venezuela's extremely high inflation or economies with unusually large trade shares.

**Do economies with higher values on one indicator generally rank higher on another?**
Spearman focuses on ranks rather than the size of the values. It captures relationships that consistently increase or decrease, even when they do not form a straight line. It is generally less sensitive to extreme numerical values.

In [275]:
spearman_correlations = plot_df[cols].corr(method="spearman")

print("Pearson correlations:")
display(correlations.round(3))

print("Spearman correlations:")
display(spearman_correlations.round(3))


Pearson correlations:


,gdp_per_capita,inflation,imports,exports
gdp_per_capita,1.000,-0.053,0.272,0.484
inflation,-0.053,1.000,-0.099,0.007
imports,0.272,-0.099,1.000,0.848
exports,0.484,0.007,0.848,1.000


Spearman correlations:


,gdp_per_capita,inflation,imports,exports
gdp_per_capita,1.000,-0.294,0.239,0.550
inflation,-0.294,1.000,-0.224,-0.309
imports,0.239,-0.224,1.000,0.722
exports,0.550,-0.309,0.722,1.000


The Pearson correlation results show that imports and exports have the strongest relationship (0.85). Economies that import more relative to their GDP also tend to export more. GDP per capita is moderately associated with exports (0.48), while inflation shows little linear relationship with the other indicators.

Because the dataset contains some extreme values, Spearman correlations were also examined. The results suggest that certain relationships, particularly those involving inflation, differ when economies are compared by their rankings rather than their actual values.

For example, inflation and exports have almost no linear relationship under Pearson correlation (0.01), but a modest negative relationship under Spearman correlation (−0.31).

These differences highlight the importance of considering how unusual observations may influence the analysis. Pearson correlation remains the primary measure because the subsequent Mahalanobis distance calculation uses the covariance structure of the original indicators.

###3. Calculate the distance measure using the Mahalanobis distance.
- Explain why you use the Mahalanobis distance over the Euclidean distance. Stata offers some canned programs including one to calcuate the Mahalanobis distance caled mahapick. This doesn't come with the standard installation so you need to first install it.
- The command to compute distance is below:
mahascores gdp_per_cap imports exports, idvar(country) genfile(mahadist) compute_invcov
- This creates a new file called "mahadist" which has all the distance scores.

In [276]:
# Economies at each observed extreme
indicators = ['gdp_per_capita', 'inflation', 'imports', 'exports']

extremes = []
for col in indicators:
    # Find the row with the minimum value for this indicator
    min_row = wdi_df.loc[wdi_df[col].idxmin()]
    # Find the row with the maximum value for this indicator
    max_row = wdi_df.loc[wdi_df[col].idxmax()]

    extremes.append({
        'Indicator': col,
        'Min Country': min_row['country'],
        'Min Value': min_row[col],
        'Max Country': max_row['country'],
        'Max Value': max_row[col]
    })

extremes_df = pd.DataFrame(extremes)
display(extremes_df)

,Indicator,Min Country,Min Value,Max Country,Max Value
0,gdp_per_capita,Central African Republic,401.825822,Luxembourg,107591.899505
1,inflation,Naoero,-4.603271,"Venezuela, RB",21909.342668
2,imports,Cuba,10.607517,"Hong Kong SAR, China",176.092206
3,exports,Nepal,7.779936,Luxembourg,206.580146


Mahalanobis Distance

In [277]:
# 1. Select the indicators to use for distance calculation
features = ['gdp_per_capita', 'inflation', 'imports', 'exports']

# 2. Filter out rows that have NaN values in our key features
wdi_clean = wdi_df.dropna(subset=features).copy()

# 3. Extract values and compute the inverse covariance matrix
x = wdi_clean[features].values
cov_matrix = np.cov(x, rowvar=False)
inv_cov_matrix = np.linalg.inv(cov_matrix)

# 4. Locate the vector for the United States
us_row = wdi_clean[wdi_clean['country'] == 'United States']
if us_row.empty:
    us_row = wdi_clean[wdi_clean['country'].str.contains('United States', case=False, na=False)]

us_vector = us_row[features].values[0]

# 5. Calculate Mahalanobis distance from US to each country
mahalanobis_distances = []
for idx, row in wdi_clean.iterrows():
    country_vector = row[features].values
    # Calculate distance
    d = distance.mahalanobis(us_vector, country_vector, inv_cov_matrix)
    mahalanobis_distances.append(d)

# Add distance scores to the cleaned dataframe
wdi_clean['mahalanobis_distance'] = mahalanobis_distances

# Save the result back to wdi_df
wdi_df = wdi_clean

# Show the dataframe sorted by distance to find the closest countries to the US
print("Mahalanobis distances successfully calculated (non-NaN)!")
display(wdi_df.sort_values(by='mahalanobis_distance').head(10))

Mahalanobis distances successfully calculated (non-NaN)!


,country,exports,gdp_per_capita,imports,inflation,mahalanobis_distance
206,United States,11.789197,61047.964824,14.469283,1.650466,0.000000
10,Australia,24.062480,59181.299790,21.939203,3.476600,0.557496
147,Norway,35.695052,79937.509789,33.185191,-0.477791,0.979035
205,United Kingdom,31.902860,47866.914035,33.100706,2.461330,1.237178
88,Iceland,44.713862,57118.845278,39.078504,4.836789,1.344622
35,Canada,32.352626,45100.336806,33.818716,1.546199,1.380437
98,Japan,17.037797,36837.258365,17.320500,0.782577,1.490063
141,New Zealand,27.320284,40615.726695,27.140962,3.299628,1.498499
187,Sweden,48.288658,53024.409949,44.462481,2.425887,1.589507
159,Qatar,52.188860,63430.460761,37.857516,-4.455914,1.592003


####4. Summarize the Mahalanobis distance measure between the United States and other countries in the world.
Which 3 countries does the United States have the smallest distance and largest distance with?

* Use the "keep if" command to restrict the above data for just distance between US and all other countries.


In [278]:
# Filter out the US itself to find the closest countries
non_us_df = wdi_df[wdi_df['country'] != 'United States']

# Get the 5 closest countries
closest_5 = non_us_df.sort_values(by='mahalanobis_distance').head(5)

# Get the 5 furthest countries
furthest_5 = non_us_df.sort_values(by='mahalanobis_distance', ascending=False).head(5)

print("=== Top 5 Countries Closest to the United States ===")
display(closest_5[['country', 'gdp_per_capita', 'inflation', 'imports', 'exports', 'mahalanobis_distance']])

print("\n=== Top 5 Countries Furthest from the United States ===")
display(furthest_5[['country', 'gdp_per_capita', 'inflation', 'imports', 'exports', 'mahalanobis_distance']])

=== Top 5 Countries Closest to the United States ===


,country,gdp_per_capita,inflation,imports,exports,mahalanobis_distance
10,Australia,59181.299790,3.476600,21.939203,24.062480,0.557496
147,Norway,79937.509789,-0.477791,33.185191,35.695052,0.979035
205,United Kingdom,47866.914035,2.461330,33.100706,31.902860,1.237178
88,Iceland,57118.845278,4.836789,39.078504,44.713862,1.344622
35,Canada,45100.336806,1.546199,33.818716,32.352626,1.380437



=== Top 5 Countries Furthest from the United States ===


,country,gdp_per_capita,inflation,imports,exports,mahalanobis_distance
210,"Venezuela, RB",1759.055474,21909.342668,10.933114,47.195726,13.837835
54,Djibouti,2781.943501,0.449845,154.221366,166.717864,7.625096
86,"Hong Kong SAR, China",44195.274114,2.044955,176.092206,177.651315,6.589756
116,Luxembourg,107591.899505,0.916369,176.080635,206.580146,6.266985
164,San Marino,43525.403980,0.914095,143.267534,163.997237,6.034089


In [279]:
X = plot_df[cols].to_numpy(dtype=float)
cov_matrix = np.cov(X, rowvar=False)
inverse_covariance = np.linalg.inv(cov_matrix)

# 2. Locate the United States reference values
us_vector = plot_df.loc[
    plot_df["country"] == "United States", cols
].iloc[0].to_numpy(dtype=float)

# 3. Compute distance metric mapping
plot_df["distance_us"] = [
    distance.mahalanobis(row, us_vector, inverse_covariance)
    for row in X
]

# 4. Sort and select top/bottom 5 countries, including the US baseline
ranked = plot_df.sort_values("distance_us")
near = ranked.head(6) # US + 5 actual closest
far = ranked.tail(5) # 5 furthest

In [280]:
# --- Visual 1: Closest Economies ---
fig_closest = go.Figure()
for _, row in near.iterrows():
    fig_closest.add_trace(go.Scatter(
        x=[0, row["distance_us"]],
        y=[row["country"], row["country"]],
        mode="lines",
        line=dict(color="lightgray", width=3),
        showlegend=False,
        hoverinfo="skip"
    ))

fig_closest.add_trace(go.Scatter(
    x=near["distance_us"],
    y=near["country"],
    mode="markers+text",
    marker=dict(color="#277C69", size=14, line=dict(width=1.5, color="black")),
    text=near["distance_us"].apply(lambda val: f" {val:.2f}"),
    textposition="middle right",
    hovertemplate="<b>%{y}</b><br>Mahalanobis Distance: %{x:.2f}<extra></extra>",
    showlegend=False
))

fig_closest.update_layout(
    title="Economic Similarity: Closest Countries with E to the United States",
    xaxis_title="Mahalanobis distance<br><sup>(lower score = more similar)</sup>",
    yaxis_title="Country",
    template="plotly_white",
    width=900,
    height=350,
    margin=dict(l=150, r=80, t=80, b=40),
    xaxis=dict(showgrid=True, gridcolor='LightGray')
)

# Add baseline axis lines
fig_closest.update_xaxes(showline=True, linewidth=0.1, linecolor='lightgray', mirror=True)
fig_closest.update_yaxes(showline=True, linewidth=0.1, linecolor='lightgray', mirror=True)

fig_closest.show()

In [281]:
# --- Visual 2: Most Distant Economies ---
fig_distant = go.Figure()
for _, row in far.iterrows():
    fig_distant.add_trace(go.Scatter(
        x=[0, row["distance_us"]],
        y=[row["country"], row["country"]],
        mode="lines",
        line=dict(color="lightgray", width=3),
        showlegend=False,
        hoverinfo="skip"
    ))

fig_distant.add_trace(go.Scatter(
    x=far["distance_us"],
    y=far["country"],
    mode="markers+text",
    marker=dict(color="#BA6849", size=14, line=dict(width=1.5, color="black")),
    text=far["distance_us"].apply(lambda val: f" {val:.2f}"),
    textposition="middle right",
    hovertemplate="<b>%{y}</b><br>Mahalanobis Distance: %{x:.2f}<extra></extra>",
    showlegend=False
))

fig_distant.update_layout(
    title="Economic Dissimilarity: Furthest Countries from the United States",
    xaxis_title="Mahalanobis distance<br><sup>(higher score = more distant)</sup>",
    yaxis_title="Country",
    template="plotly_white",
    width=900,
    height=350,
    margin=dict(l=150, r=80, t=80, b=40),
    xaxis=dict(showgrid=True, gridcolor='LightGray')
)

# Add baseline axis lines
fig_distant.update_xaxes(showline=True, linewidth=0.1, linecolor='lightgray', mirror=True)
fig_distant.update_yaxes(showline=True, linewidth=0.1, linecolor='lightgray', mirror=True)

fig_distant.show()